# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SIDRAATTIQUE/flyrank-machine-learning/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np
import os

HF_TOKEN = userdata.get('HF_Token')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Point to already-downloaded local files
local_path = '/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2'

LOCAL_FACT = f"read_parquet('{local_path}/fact_content_daily_performance/month=2026-0*/*.parquet')"

print("✅ Connected and local files ready")

✅ Connected and local files ready


In [3]:
from huggingface_hub import snapshot_download
from google.colab import userdata

HF_TOKEN = userdata.get('HF_Token')

print("Re-downloading Feb-March 2026 partitions...")
local_path = snapshot_download(
    repo_id='FlyRank/internship-warehouse',
    repo_type='dataset',
    token=HF_TOKEN,
    allow_patterns=[
        'fact_content_daily_performance/month=2026-02/*.parquet',
        'fact_content_daily_performance/month=2026-03/*.parquet',
    ]
)
print(f"✅ Downloaded to: {local_path}")
print(f"Save this path: {local_path}")

Re-downloading Feb-March 2026 partitions...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Downloaded to: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2
Save this path: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2


In [4]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

LOCAL_FACT = f"read_parquet('{local_path}/fact_content_daily_performance/month=2026-0*/*.parquet')"

print("Rebuilding feature frame...")

feature_frame = con.sql(f"""
    WITH features AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date >= '2026-03-01'
                THEN gsc_impressions ELSE 0 END)         AS imp_last30,
            SUM(CASE WHEN report_date < '2026-03-01'
                THEN gsc_impressions ELSE 0 END)          AS imp_prev30,
            SUM(CASE WHEN report_date >= '2026-03-01'
                THEN gsc_clicks ELSE 0 END)               AS clk_last30,
            SUM(CASE WHEN report_date < '2026-03-01'
                THEN gsc_clicks ELSE 0 END)               AS clk_prev30,
            AVG(CASE WHEN report_date >= '2026-03-01'
                THEN gsc_avg_position END)                AS pos_last30,
            AVG(CASE WHEN report_date < '2026-03-01'
                THEN gsc_avg_position END)                AS pos_prev30,
            STDDEV(gsc_avg_position)                      AS pos_volatility,
            COUNT(DISTINCT report_date)                   AS days_of_data
        FROM {LOCAL_FACT}
        WHERE report_date >= '2026-02-01'
          AND report_date <= '2026-03-31'
        GROUP BY 1, 2
        HAVING imp_prev30 >= 10
    )
    SELECT * FROM features
""").df()

feature_frame['ctr_last30'] = feature_frame['clk_last30'] / feature_frame['imp_last30'].replace(0, np.nan)
feature_frame['ctr_prev30'] = feature_frame['clk_prev30'] / feature_frame['imp_prev30'].replace(0, np.nan)
feature_frame['ctr_change'] = feature_frame['ctr_last30'] - feature_frame['ctr_prev30']
feature_frame['pos_change'] = feature_frame['pos_last30'] - feature_frame['pos_prev30']
feature_frame['is_declining'] = (
    feature_frame['imp_last30'] < 0.8 * feature_frame['imp_prev30']
).astype(int)

print(f"✅ Rows: {len(feature_frame):,} | Clients: {feature_frame['client_hash_id'].nunique()}")
display(feature_frame.head())

Rebuilding feature frame...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Rows: 119,340 | Clients: 42


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,clk_prev30,pos_last30,pos_prev30,pos_volatility,days_of_data,ctr_last30,ctr_prev30,ctr_change,pos_change,is_declining
0,client_3ffa76342f366962,content_78fce5fb92fdf949,29.0,15.0,0.0,0.0,4.583333,4.111111,2.729518,49,0.0,0.000000,0.000000,0.472222,0
1,client_3ffa76342f366962,content_cae1d5374958a649,43.0,96.0,0.0,0.0,7.352381,5.262333,2.871747,58,0.0,0.000000,0.000000,2.090047,1
2,client_3ffa76342f366962,content_dd66eecf9626cab8,255.0,235.0,0.0,0.0,5.853611,6.407819,1.834977,59,0.0,0.000000,0.000000,-0.554208,0
3,client_3ffa76342f366962,content_c51f1e8ef5502177,15.0,18.0,0.0,0.0,7.766667,6.961538,2.712813,47,0.0,0.000000,0.000000,0.805128,0
4,client_3ffa76342f366962,content_0674cc4ae0f68a90,11.0,74.0,0.0,1.0,6.333333,8.063910,2.505196,54,0.0,0.013514,-0.013514,-1.730576,1


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

##  Two paper findings + my methodology questions

I read the FlyRank SEO Research Paper (March 2026) and selected
two findings to examine constructively.

---

### Finding 1: Position is associated with CTR in a non-linear way
(Pages ranking in positions 1-3 showed dramatically higher CTR
than pages in positions 4-10)

**My methodology question:**
Where does the CTR label come from, and is it measured at the
page level or the query level? If CTR is averaged across all
queries a page ranks for, then a page ranking #1 for one query
and #20 for ten others could show a misleadingly "good" average
position while actually having poor visibility. I would ask:
was the position-CTR relationship measured per query-page pair
(more precise) or per page aggregate (simpler but noisier)?

This is not a criticism — aggregating is a valid, common choice
for this type of analysis. I raise it because my own work uses
the same aggregation (gsc_avg_position), so the same caveat
applies to my model.

---

### Finding 2: Content refresh is associated with measurable
impression recovery in some cases

**My methodology question:**
Does the validation design support the causal claim, or is this
an observed correlation? Specifically: was there a control group
of similar pages that were NOT refreshed, measured over the same
time window? Without a control, it is difficult to rule out that
impressions recovered naturally (e.g., seasonal demand returning)
rather than because of the refresh.

Again, this is the same limitation my own model faces — I can
observe that pages with declining impressions TEND to stabilize,
but I cannot prove the refresh caused the recovery. The paper
appears to acknowledge this with careful language, which is the
right approach. My capstone will use the same careful framing.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

##  My model under an honest split (before/after)

In Week 5, I already used GroupShuffleSplit by client_hash_id.
This section shows the before/after: what would happen with a
naive random split vs the grouped split, to demonstrate why
the grouped split is more honest.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

feature_cols = ['ctr_change', 'pos_volatility', 'pos_change', 'imp_prev30', 'days_of_data']
model_data = feature_frame.dropna(subset=feature_cols + ['is_declining']).copy()

X = model_data[feature_cols]
y = model_data['is_declining']
groups = model_data['client_hash_id']

def evaluate_model(X_tr, X_te, y_tr, y_te, label):
    m = RandomForestClassifier(
        n_estimators=100, max_depth=6,
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    m.fit(X_tr, y_tr)
    preds = m.predict(X_te)
    return {
        'Split Type': label,
        'Train rows': len(X_tr),
        'Test rows':  len(X_te),
        'Accuracy':   round(accuracy_score(y_te, preds), 3),
        'Precision':  round(precision_score(y_te, preds, zero_division=0), 3),
        'Recall':     round(recall_score(y_te, preds, zero_division=0), 3),
        'F1':         round(f1_score(y_te, preds, zero_division=0), 3),
    }

results = []

# BEFORE: Naive random split (the wrong way)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)
results.append(evaluate_model(X_tr, X_te, y_tr, y_te, 'Naive Random Split (BEFORE)'))

# AFTER: Grouped split by client (the honest way)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_tr_g = X.iloc[train_idx]
X_te_g = X.iloc[test_idx]
y_tr_g = y.iloc[train_idx]
y_te_g = y.iloc[test_idx]
results.append(evaluate_model(X_tr_g, X_te_g, y_tr_g, y_te_g, 'Grouped by Client (AFTER)'))

comparison_df = pd.DataFrame(results)
print("BEFORE vs AFTER: Split Design Comparison")
display(comparison_df)

print("\nInterpretation:")
print("If random split F1 > grouped split F1, the model was")
print("memorizing client patterns, not learning general rules.")
print("The grouped split is the honest number to report.")

BEFORE vs AFTER: Split Design Comparison


,Split Type,Train rows,Test rows,Accuracy,Precision,Recall,F1
0,Naive Random Split (BEFORE),83620,27874,0.614,0.290,0.653,0.402
1,Grouped by Client (AFTER),75002,36492,0.563,0.318,0.612,0.419



Interpretation:
If random split F1 > grouped split F1, the model was
memorizing client patterns, not learning general rules.
The grouped split is the honest number to report.


In [7]:
# Show the before/after numbers clearly
print("KEY FINDING:")
print("=" * 50)
naive_f1 = comparison_df[comparison_df['Split Type'].str.contains('Naive')]['F1'].values[0]
grouped_f1 = comparison_df[comparison_df['Split Type'].str.contains('Grouped')]['F1'].values[0]

if naive_f1 > grouped_f1:
    diff = round(naive_f1 - grouped_f1, 3)
    print(f"Naive random split F1:  {naive_f1}")
    print(f"Grouped split F1:       {grouped_f1}")
    print(f"Difference:             -{diff}")
    print()
    print("The naive split INFLATED performance by", diff)
    print("This confirms the model was partially memorizing")
    print("client-specific patterns, not just learning general rules.")
    print("The grouped split number is the honest one to report.")
else:
    diff = round(grouped_f1 - naive_f1, 3)
    print(f"Naive random split F1:  {naive_f1}")
    print(f"Grouped split F1:       {grouped_f1}")
    print(f"Difference:             +{diff}")
    print()
    print("The grouped split performed comparably to the random split.")
    print("This is a good sign — it suggests the model learned")
    print("general patterns rather than client-specific memorization.")
    print("The grouped split is still the honest number to report.")

KEY FINDING:
Naive random split F1:  0.402
Grouped split F1:       0.419
Difference:             +0.017

The grouped split performed comparably to the random split.
This is a good sign — it suggests the model learned
general patterns rather than client-specific memorization.
The grouped split is still the honest number to report.


In [8]:
# Verify claims with actual numbers from previous runs
print("CLAIM VERIFICATION — Numbers that back the rewrites")
print("=" * 55)
print()
print("Claim 1: F1 comparison")
print(f"  Week-4 Rule Baseline F1:        0.155")
print(f"  RF Balanced F1:                 0.419")
print(f"  RF Balanced Precision:          0.318")
print(f"  RF Balanced Recall:             0.616")
print(f"  Source: w05_model.ipynb, grouped test set")
print()
print("Claim 2: Feature importance")
print(f"  ctr_change importance:          0.068")
print(f"  pos_volatility importance:      0.050")
print(f"  Measured via: permutation importance, 10 repeats")
print(f"  Source: w05_model.ipynb, balanced RF on grouped test set")
print()
print("Claim 3: Split design")
print(f"  Train clients:  27")
print(f"  Test clients:   10")
print(f"  Total clients:  37")
print(f"  Split method:   GroupShuffleSplit(test_size=0.25)")
print()
print("ALL CLAIMS ARE BACKED BY CODE IN w05_model.ipynb")
print("All language has been rewritten to be directional,")
print("observed, and decision-support only.")

CLAIM VERIFICATION — Numbers that back the rewrites

Claim 1: F1 comparison
  Week-4 Rule Baseline F1:        0.155
  RF Balanced F1:                 0.419
  RF Balanced Precision:          0.318
  RF Balanced Recall:             0.616
  Source: w05_model.ipynb, grouped test set

Claim 2: Feature importance
  ctr_change importance:          0.068
  pos_volatility importance:      0.050
  Measured via: permutation importance, 10 repeats
  Source: w05_model.ipynb, balanced RF on grouped test set

Claim 3: Split design
  Train clients:  27
  Test clients:   10
  Total clients:  37
  Split method:   GroupShuffleSplit(test_size=0.25)

ALL CLAIMS ARE BACKED BY CODE IN w05_model.ipynb
All language has been rewritten to be directional,
observed, and decision-support only.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage audit

For each feature used in my Week-5 model, I audit whether it
could have "leaked" future information into the training data.

In [6]:
leakage_audit = pd.DataFrame({
    'Feature': [
        'ctr_change',
        'pos_volatility',
        'pos_change',
        'imp_prev30',
        'days_of_data'
    ],
    'Time window used': [
        'Feb vs March 2026',
        'Full Feb-March window',
        'Feb vs March 2026',
        'February 2026 only',
        'Feb-March 2026 count'
    ],
    'Decision moment': [
        'March 31, 2026',
        'March 31, 2026',
        'March 31, 2026',
        'March 31, 2026',
        'March 31, 2026'
    ],
    'Leakage risk': [
        'LOW — both periods are before decision date',
        'LOW — historical volatility only',
        'LOW — both periods are before decision date',
        'LOW — February is before March decision window',
        'LOW — count of historical days only'
    ],
    'Label derived?': [
        'NO — CTR is an independent signal',
        'NO — position is an independent signal',
        'NO — position change is an independent signal',
        'NO — previous impressions are independent',
        'NO — data availability count only'
    ]
})

print("LEAKAGE AUDIT: All features in Week-5 model")
display(leakage_audit)

print("\nCONCLUSION: No leakage detected.")
print("All features use data available at the decision moment (March 31).")
print("The label (is_declining) is defined using March impressions,")
print("which is the OUTCOME being predicted — not used as a feature input.")
print("\nNote from Week-4: I already demonstrated the leakage trap by")
print("adding and removing LEAKED_future_clicks. That column was deleted")
print("and does not appear in any model in this notebook.")

LEAKAGE AUDIT: All features in Week-5 model


,Feature,Time window used,Decision moment,Leakage risk,Label derived?
0,ctr_change,Feb vs March 2026,"March 31, 2026",LOW — both periods are before decision date,NO — CTR is an independent signal
1,pos_volatility,Full Feb-March window,"March 31, 2026",LOW — historical volatility only,NO — position is an independent signal
2,pos_change,Feb vs March 2026,"March 31, 2026",LOW — both periods are before decision date,NO — position change is an independent signal
3,imp_prev30,February 2026 only,"March 31, 2026",LOW — February is before March decision window,NO — previous impressions are independent
4,days_of_data,Feb-March 2026 count,"March 31, 2026",LOW — count of historical days only,NO — data availability count only



CONCLUSION: No leakage detected.
All features use data available at the decision moment (March 31).
The label (is_declining) is defined using March impressions,
which is the OUTCOME being predicted — not used as a feature input.

Note from Week-4: I already demonstrated the leakage trap by
adding and removing LEAKED_future_clicks. That column was deleted
and does not appear in any model in this notebook.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim rewrite

### Original claim (Week-5 notebook):
"The balanced Random Forest outperforms the Week-4 rule baseline."

### Why it needs rewriting:
"Outperforms" implies general superiority. But the model only
outperforms on F1 (0.419 vs 0.155) while underperforming on
precision (0.318 vs 1.000) and accuracy (56.2% vs 76.5%).
"Outperforms" without qualification is an overclaim.

### Rewritten claim (safe language):
"On the F1 metric — which balances finding opportunities
(recall) with avoiding false alarms (precision) — the balanced
Random Forest (F1 = 0.419) outperformed the Week-4 rule
(F1 = 0.155) on the grouped test set. This improvement came at
the cost of lower precision (0.318 vs 1.000): the model flags
more pages overall but accepts more false positives. Which model
is preferable depends on the content team's capacity to review
flagged pages."

---

### Original claim (Week-5 notebook):
"ctr_change is the strongest signal for detecting declining pages."

### Why it needs rewriting:
This was observed on one time window (Feb-March 2026) across
37 clients. It has not been validated on other time windows
or a held-out test period (June 2026).

### Rewritten claim (safe language):
"In this dataset and time window, permutation importance
measured on the grouped test set suggests ctr_change was the
most associated feature with the is_declining label (importance
= 0.068). This is a directional, observed finding on one
two-month window and has not been validated on the sealed
June 2026 test period."

---

### Original claim (Week-5 notebook):
"The grouped split proves the model generalizes across clients."

### Why it needs rewriting:
A single grouped split tests on 10 clients. "Proves" is too
strong for a single split on 10 test clients.

### Rewritten claim (safe language):
"The grouped split (27 train clients, 10 test clients) provides
a more conservative and honest estimate of generalization than
a random row-level split. It suggests the model may generalize
across clients, but a single split is not sufficient to prove
this conclusively — cross-validation across multiple client
folds would be needed for a stronger claim."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.